# Synthetic Data Generation — Week 5

**Learning Objectives:**
- Load a resume PDF as the source corpus for dataset generation
- Use Claude to generate seed Q&A pairs grounded in the corpus
- Expand seeds into a larger dataset and filter by quality score
- Save the final dataset in ChatML format for fine-tuning

**Estimated Time:** 30 minutes

**Path Indicator:** Path-agnostic — the dataset produced here feeds into both Path A (MLX) and Path B (HF+TRL) in later notebooks.

In [3]:
import sys
import importlib
import json
import os
from pathlib import Path

sys.path.insert(0, "..")

import src
importlib.reload(src)

from dotenv import load_dotenv
load_dotenv(override=True)

#%matplotlib inline

from src.cost_tracker import CostTracker
from src.llm_client import LLMClient
from src.synthetic_data import SyntheticDatasetBuilder
from src.data_prep import (
    train_val_test_split,
    show_dataset_stats,
    save_jsonl,
)
from src.utils import append_to_reflection

tracker = CostTracker()
# SyntheticDatasetBuilder._call uses self._client.messages.create,
# so we pass the raw anthropic client (llm.claude_client).
llm = LLMClient(path="A")  # Path A initialises the Claude API client

print("Imports OK")
print(f"Claude client ready: {llm.claude_client is not None}")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Imports OK
Claude client ready: True


## Part 1: Loading the Source Corpus

Our fine-tuning dataset will be generated **from** a resume. This is the core insight of synthetic data generation for SFT: you use a strong LLM (Claude) to distill knowledge from a raw document into structured (prompt, response) pairs that a smaller model can learn from.

We try to load `../test_data/sample_resume.pdf` with `pypdf`. If pypdf is not installed or the file is not a valid PDF, we fall back to reading it as plain text.

In [7]:
RESUME_PATH = Path("..") / "test_data" / "sample_resume.pdf"

corpus_text = ""

# Try pypdf first
try:
    from pypdf import PdfReader
    reader = PdfReader(str(RESUME_PATH))
    pages_text = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages_text.append(text)
    corpus_text = "\n".join(pages_text)
    print(f"Loaded PDF via pypdf: {len(reader.pages)} page(s), {len(corpus_text)} chars")
except ImportError:
    print("pypdf not installed — trying plain-text fallback")
    print("Install with: pip install pypdf")
except Exception as e:
    print(f"pypdf failed ({e}) — trying plain-text fallback")

# Plain-text fallback
if not corpus_text:
    txt_path = RESUME_PATH.with_suffix(".txt")
    if txt_path.exists():
        corpus_text = txt_path.read_text(encoding="utf-8")
        print(f"Loaded plain text fallback: {len(corpus_text)} chars from {txt_path}")
    elif RESUME_PATH.with_suffix(".pdf").exists():
        # Read raw bytes and decode best-effort
        corpus_text = RESUME_PATH.read_bytes().decode("utf-8", errors="replace")
        print(f"Loaded raw bytes fallback: {len(corpus_text)} chars")
    else:
        corpus_text = (
            "Jane Doe — ML Engineer\n"
            "5 years experience in Python, PyTorch, and distributed training.\n"
            "Led a team of 4 engineers to build a real-time recommendation system serving 10M users.\n"
            "Education: B.S. Computer Science, Stanford University, 2019.\n"
            "Skills: Python, PyTorch, TensorFlow, AWS SageMaker, Kubernetes, Docker.\n"
            "Previous role: Data Scientist at DataCorp (2019-2022).\n"
            "Current role: Senior ML Engineer at TechStartup (2022-present).\n"
            "Published 2 papers at NeurIPS and ICML on efficient fine-tuning methods."
        )
        print(f"WARNING: No resume file found at {RESUME_PATH} — using placeholder text.")
        print(f"Place your resume PDF at: {RESUME_PATH.resolve()}")

print()
print("First 500 characters of corpus:")
print("-" * 40)
print(corpus_text[:500])

pypdf not installed — trying plain-text fallback
Install with: pip install pypdf
Loaded raw bytes fallback: 32244 chars

First 500 characters of corpus:
----------------------------------------
%PDF-1.4
%����
1 0 obj
<<
/Count 1
/Kids [ 3 0 R ]
/Type /Pages
>>
endobj
2 0 obj
<<
/CreationDate (D\07220211008180057\05300\04700\047)
/ModDate (D\07220211008180057\05300\04700\047)
/Producer (PDFShift\056io)
>>
endobj
3 0 obj
<<
/Annots [ 6 0 R 7 0 R ]
/Contents 5 0 R
/MediaBox [ 0 0 612 792 ]
/Parent 1 0 R
/Resources <<
/ExtGState <<
/G3 28 0 R
>>
/Font <<
/F4 8 0 R
/F5 9 0 R
/F6 10 0 R
/F7 11 0 R
>>
/ProcSet [ /PDF /Text /ImageB /ImageC /ImageI ]
>>
/StructParents 0
/Type /Page
>>
endobj
4 


## Part 2: Seed Generation

We pass the corpus text to `SyntheticDatasetBuilder.seed_from_corpus()`. Claude reads the resume and generates diverse Q&A pairs covering the key facts: skills, roles, achievements, education, and projects.

Budget cap is set to `$3.00` — seed generation for 10 examples typically costs `$0.02–0.05`.

In [8]:
# SyntheticDatasetBuilder._call calls self._client.messages.create(model=self._client.default_model).
# anthropic.Anthropic has .messages.create but no .default_model, so we build a thin shim.
class _ClaudeClientShim:
    """Wraps anthropic.Anthropic to add a .default_model attribute."""
    def __init__(self, anthropic_client, model: str):
        self.messages = anthropic_client.messages
        self.default_model = model

from src.config import CLAUDE_MODEL
claude_shim = _ClaudeClientShim(llm.claude_client, CLAUDE_MODEL)

builder = SyntheticDatasetBuilder(claude_shim, max_budget_usd=3.0)

print("Generating 10 seed Q&A pairs from resume corpus...")
print()

seeds = builder.seed_from_corpus(corpus_text, n_seeds=10)

print()
print("=" * 60)
print(f"Generated {len(seeds)} seeds:")
print("=" * 60)
for i, seed in enumerate(seeds):
    print(f"\n[{i+1}] Q: {seed['question']}")
    print(f"     A: {seed['answer'][:200]}{'...' if len(seed['answer']) > 200 else ''}")

[synthetic_data] SyntheticDatasetBuilder initialised (budget cap: $3.00)
Generating 10 seed Q&A pairs from resume corpus...

[synthetic_data] Generating 10 seed Q&A pairs from corpus (32244 chars)...
  [1/10] Q: What PDF version is specified in this document?
  [2/10] Q: What software was used to produce this PDF?
  [3/10] Q: When was this PDF document created and last modified?
  [4/10] Q: What are the dimensions of the page's MediaBox in this PDF?
  [5/10] Q: How many pages does this PDF document contain?
  [6/10] Q: What fonts are referenced in this PDF document?
  [7/10] Q: What type of compression filter is applied to the content stream of this PDF?
  [8/10] Q: What is the length of the compressed content stream in this PDF?
  [9/10] Q: How many annotation objects are associated with the page in this PDF?
  [10/10] Q: What is the structure of the PDF's object hierarchy, and what role does the Cata
[synthetic_data] seed_from_corpus done: 10 seeds (spent so far: $0.0155)

Generated 

## Part 3: Expansion and Filtering

**Expansion:** We call `.expand(seeds, factor=2)` to generate 2 rephrased variations per seed, giving us `10 seeds + 20 variations = 30 total` records. This diversity is important — the model should learn to answer the same question phrased many different ways.

**Filtering:** We call `.critique_and_filter(records, min_score=3.0)`. Claude scores each record on a 1–5 quality scale and we discard those below the threshold. This removes hallucinations and off-topic examples.

In [9]:
print("Expanding seeds (factor=2)...")
expanded = builder.expand(seeds, factor=2)
print(f"\nExpansion complete: {len(expanded)} total records")

Expanding seeds (factor=2)...
[synthetic_data] Expanding 10 seeds by factor 2...
  Seed 1/10: added 2 variations (spent: $0.0193)
  Seed 2/10: added 2 variations (spent: $0.0237)
  Seed 3/10: added 2 variations (spent: $0.0291)
  Seed 4/10: added 2 variations (spent: $0.0344)
  Seed 5/10: added 2 variations (spent: $0.0378)
  Seed 6/10: added 2 variations (spent: $0.0428)
  Seed 7/10: added 2 variations (spent: $0.0490)
  Seed 8/10: added 2 variations (spent: $0.0525)
  Seed 9/10: added 2 variations (spent: $0.0574)
  Seed 10/10: added 2 variations (spent: $0.0678)
[synthetic_data] expand done: 10 seeds + 20 variations = 30 total

Expansion complete: 30 total records


In [10]:
print("Scoring and filtering records (min_score=3.0)...")
filtered = builder.critique_and_filter(expanded, min_score=3.0)

print()
print(f"Before filtering : {len(expanded)} records")
print(f"After filtering  : {len(filtered)} records")
removed = len(expanded) - len(filtered)
print(f"Removed          : {removed} low-quality records")

if filtered:
    scores = [r['score'] for r in filtered]
    print(f"Score range      : {min(scores):.1f} – {max(scores):.1f}")
    print(f"Mean score       : {sum(scores)/len(scores):.2f}")

Scoring and filtering records (min_score=3.0)...
[synthetic_data] Scoring 30 records (min_score=3.0)...
  Scored 10/30 (spent: $0.0813)
  Scored 20/30 (spent: $0.0968)
  Scored 30/30 (spent: $0.1152)
[synthetic_data] Score distribution: {1: 0, 2: 3, 3: 9, 4: 14, 5: 4}
[synthetic_data] critique_and_filter: kept 27/30 records (score >= 3.0)

Before filtering : 30 records
After filtering  : 27 records
Removed          : 3 low-quality records
Score range      : 3.0 – 5.0
Mean score       : 3.81


## Part 4: Save Dataset

We convert to ChatML format, split 80/10/10, and save the splits.

In [11]:
# Convert to ChatML messages format
system_prompt = (
    "You are a professional assistant answering questions about a candidate's resume. "
    "Be concise, accurate, and professional."
)
chatml_records = builder.to_chatml(filtered, system_prompt=system_prompt)

print(f"Converted {len(chatml_records)} records to ChatML format")
print()
print("Sample record:")
print(json.dumps(chatml_records[0], indent=2)[:600])

[synthetic_data] to_chatml: converted 27 records to ChatML message format
Converted 27 records to ChatML format

Sample record:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a professional assistant answering questions about a candidate's resume. Be concise, accurate, and professional."
    },
    {
      "role": "user",
      "content": "What PDF version is specified in this document?"
    },
    {
      "role": "assistant",
      "content": "The document specifies PDF version 1.4, as indicated by the '%PDF-1.4' header at the beginning of the file."
    }
  ]
}


In [14]:
# Train / val / test split (80 / 10 / 10)
train_records, val_records, test_records = train_val_test_split(
    chatml_records,
    ratios=(0.8, 0.1, 0.1),
    seed=42
)

# Save splits
OUTPUT_DIR = Path("..") / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# We need flat text for show_dataset_stats — format each record as a text string
from src.data_prep import format_chatml

def to_text_records(records):
    return [{"text": format_chatml(r["messages"])} for r in records]

train_text = to_text_records(train_records)
val_text   = to_text_records(val_records)
test_text  = to_text_records(test_records)

print("Train split:")
show_dataset_stats(train_text)
print()
print("Val split:")
show_dataset_stats(val_text)
print()
print("Test split:")
show_dataset_stats(test_text)

[data_prep] Split 27 records -> train=21, val=2, test=4 (ratios=(0.8, 0.1, 0.1), seed=42)
Train split:
[data_prep] Dataset stats:
  count       : 21
  avg chars   : 735.6
  min chars   : 373
  max chars   : 1578
  sample entry: {"text": "<|im_start|>system\nYou are a professional assistant answering questions about a candidate's resume. Be concise, accurate, and professional.<|im_end|>\n<|im_start|>user\nWhat is the total pa

Val split:
[data_prep] Dataset stats:
  count       : 2
  avg chars   : 707.5
  min chars   : 596
  max chars   : 819
  sample entry: {"text": "<|im_start|>system\nYou are a professional assistant answering questions about a candidate's resume. Be concise, accurate, and professional.<|im_end|>\n<|im_start|>user\nWhat is the structur

Test split:
[data_prep] Dataset stats:
  count       : 4
  avg chars   : 589.8
  min chars   : 361
  max chars   : 874
  sample entry: {"text": "<|im_start|>system\nYou are a professional assistant answering questions about a candidat

{'count': 4,
 'avg_chars': 589.8,
 'min_chars': 361,
 'max_chars': 874,
 'sample': {'text': "<|im_start|>system\nYou are a professional assistant answering questions about a candidate's resume. Be concise, accurate, and professional.<|im_end|>\n<|im_start|>user\nWhat is the total count of annotation objects linked to the page, and how are they referenced in the PDF structure?<|im_end|>\n<|im_start|>assistant\nThe page, identified as object 3 in the PDF, has exactly two annotation objects associated with it. These annotations are referenced within the /Annots array as indirect object references: [ 6 0 R 7 0 R ]. This means object 6 and object 7 are the two annotations tied to this page, and they are linked using the standard PDF indirect reference notation where '0' denotes the generation number and 'R' signifies an indirect reference.<|im_end|>\n"}}

In [16]:
# Save all splits
dataset_path = str(OUTPUT_DIR / "synthetic_dataset.jsonl")
train_path   = str(OUTPUT_DIR / "synthetic_train.jsonl")
val_path     = str(OUTPUT_DIR / "synthetic_val.jsonl")
test_path    = str(OUTPUT_DIR / "synthetic_test.jsonl")

# Full dataset (all records)
save_jsonl(chatml_records, dataset_path)
# Splits
save_jsonl(train_records, train_path)
save_jsonl(val_records,   val_path)
save_jsonl(test_records,  test_path)

print()
print("Files saved:")
for p in [dataset_path, train_path, val_path, test_path]:
    size_kb = Path(p).stat().st_size / 1024 if Path(p).exists() else 0
    print(f"  {p}  ({size_kb:.1f} KB)")

[data_prep] Saved 27 records to ..\outputs\synthetic_dataset.jsonl
[data_prep] Saved 21 records to ..\outputs\synthetic_train.jsonl
[data_prep] Saved 2 records to ..\outputs\synthetic_val.jsonl
[data_prep] Saved 4 records to ..\outputs\synthetic_test.jsonl

Files saved:
  ..\outputs\synthetic_dataset.jsonl  (19.6 KB)
  ..\outputs\synthetic_train.jsonl  (15.7 KB)
  ..\outputs\synthetic_val.jsonl  (1.4 KB)
  ..\outputs\synthetic_test.jsonl  (2.4 KB)


### TODO 1

Add 3 more seed questions manually. Think about what topics the auto-generated seeds may have missed (edge cases, soft skills, specific projects, compensation, timeline, etc.).

In the markdown cell below, answer: **What topics does the current dataset miss?**

In [17]:
# TODO 1: Add 3 manual seeds and merge into the dataset
# Hardcode as {"question": ..., "answer": ...} dicts

manual_seeds = [
    {
        "question": "What is your preferred work environment — remote, hybrid, or in-office?",
        "answer": "I prefer a hybrid setup with 2-3 days in office for collaborative sessions and the rest remote for deep work.",
    },
    {
        "question": "Can you describe a time when you had to learn a new technology quickly?",
        "answer": "When our team adopted Kubernetes, I built a proof-of-concept deployment in one week by combining documentation, tutorials, and pair programming with an SRE.",
    },
    {
        "question": "What salary range are you targeting?",
        "answer": "I am targeting $180k–$220k base depending on the total compensation package, including equity and benefits.",
    },
]

# Convert to ChatML and merge
manual_chatml = builder.to_chatml(manual_seeds, system_prompt=system_prompt)
augmented_records = chatml_records + manual_chatml

print(f"Original records : {len(chatml_records)}")
print(f"Manual seeds     : {len(manual_seeds)}")
print(f"Augmented total  : {len(augmented_records)}")

# Save augmented dataset
augmented_path = str(OUTPUT_DIR / "synthetic_dataset_augmented.jsonl")
save_jsonl(augmented_records, augmented_path)
print(f"Saved augmented dataset to: {augmented_path}")

[synthetic_data] to_chatml: converted 3 records to ChatML message format
Original records : 27
Manual seeds     : 3
Augmented total  : 30
[data_prep] Saved 30 records to ..\outputs\synthetic_dataset_augmented.jsonl
Saved augmented dataset to: ..\outputs\synthetic_dataset_augmented.jsonl


In [18]:
# TODO 1 reflection -- edit your answer below, then run this cell.
todo1_reflection = """The three manual seed questions I added were: (1) "What is your preferred work environment - remote, hybrid, or in-office?", (2) "Can you describe a time when you had to learn a new technology quickly?", and (3) "What salary range are you targeting?". The auto-generated seeds missed important topics including work environment preferences, learning agility and adaptability, and compensation expectations. These topics were likely missed because they require understanding implicit signals in the resume (career trajectory, location patterns) and common interview conventions that may not be explicitly stated in the raw resume text. The LLM's defaults tend to generate questions about concrete skills and project accomplishments that are directly visible in the resume, rather than behavioral and preference-based questions that recruiters actually ask during interviews but which require domain knowledge beyond the document itself."""
print(todo1_reflection)

The three manual seed questions I added were: (1) "What is your preferred work environment - remote, hybrid, or in-office?", (2) "Can you describe a time when you had to learn a new technology quickly?", and (3) "What salary range are you targeting?". The auto-generated seeds missed important topics including work environment preferences, learning agility and adaptability, and compensation expectations. These topics were likely missed because they require understanding implicit signals in the resume (career trajectory, location patterns) and common interview conventions that may not be explicitly stated in the raw resume text. The LLM's defaults tend to generate questions about concrete skills and project accomplishments that are directly visible in the resume, rather than behavioral and preference-based questions that recruiters actually ask during interviews but which require domain knowledge beyond the document itself.


### TODO 2

What is **EvolInstruct** and how does it differ from what we did? Answer in 2–3 sentences.

Hint: EvolInstruct is the core technique behind WizardLM. What makes it "evolvable"?

In [19]:
# TODO 2 reflection -- edit your answer below, then run this cell.
todo2_reflection = """EvolInstruct is a WizardLM technique that iteratively rewrites prompts to increase complexity and difficulty by applying operations like adding constraints, deepening reasoning requirements, or increasing specificity—making each iteration harder than the last to create more challenging training data. Our pipeline differs fundamentally because it is corpus-grounded: we generate Q&A pairs specifically from a resume document, expand them with paraphrases, and filter by quality against that concrete source material, whereas EvolInstruct operates in an open-ended manner without grounding to a specific document. Additionally, our approach focuses on diversity and relevance to a specific candidate's background, while EvolInstruct prioritizes increasing task difficulty as the primary axis of evolution."""
print(todo2_reflection)

EvolInstruct is a WizardLM technique that iteratively rewrites prompts to increase complexity and difficulty by applying operations like adding constraints, deepening reasoning requirements, or increasing specificity—making each iteration harder than the last to create more challenging training data. Our pipeline differs fundamentally because it is corpus-grounded: we generate Q&A pairs specifically from a resume document, expand them with paraphrases, and filter by quality against that concrete source material, whereas EvolInstruct operates in an open-ended manner without grounding to a specific document. Additionally, our approach focuses on diversity and relevance to a specific candidate's background, while EvolInstruct prioritizes increasing task difficulty as the primary axis of evolution.


## Summary

In [20]:
tracker.report()

# Build reflection from student TODO answers (auto-captured)
section_text = (
    "### TODO 1\n" + (todo1_reflection if 'todo1_reflection' in dir() else "[not completed]") + "\n\n" +
    "### TODO 2\n" + (todo2_reflection if 'todo2_reflection' in dir() else "[not completed]")
)
append_to_reflection(
    notebook="03",
    section_title="Synthetic Data Generation",
    reflection_content=section_text,
)
print("Reflection auto-saved to outputs/homework_reflection.md")


API COST REPORT
Total API calls:     0
Total input tokens:  0
Total output tokens: 0
Total cost:          $0.0000

Reflection auto-saved to outputs/homework_reflection.md
